In [20]:
import polars as pl
from abc import ABC, abstractmethod
from typing import Iterable, Dict, List
from dataclasses import dataclass

### Define labels first

In [21]:
@dataclass
class Label:
    # A Label is a mapping of Node names to atomic labels
    mapping: Dict[str, pl.DataType]
    
    def matches(self, **kwargs):
        for k, v in kwargs.items():
            if k not in self.mapping:
                raise ValueError(f"Trying to match on unknown namespace {k}")
            elif self.mapping[k] != v:
                return False
        return True
    
    def drop(self, *keys):
        new_mapping = {k:v for k, v in self.mapping.items() if k not in keys}
        return Label(new_mapping)

### Define the interface for the Model Class

In [22]:
class Model(ABC):

    @abstractmethod
    def fit(self, df: pl.DataFrame, *args, **kwargs) -> None:
        raise NotImplementedError

    @abstractmethod
    def predict(self, df: pl.DataFrame, *args, **kwargs) -> pl.Series:
        raise NotImplementedError

    @property
    def labels(self) -> List[Label]:
        return [Label({'leaf': self.name})]
    
    @property
    @abstractmethod
    def name(self) -> str:
        raise NotImplementedError
    
###################################
    @property
    @abstractmethod
    def model_constructor(self):
        raise NotImplementedError
    
    def train_mask_for_label(self, label, *args, **kwargs):
        return True
    
    def test_mask_for_label(self, label, *args, **kwargs):
        return True

    @classmethod
    def __subclasshook__(cls, C):

        has_attr = lambda method: any(method in B.__dict__ for B in C.__mro__)

        if cls is Model:
            if all([has_attr(method) for method in dir(Model) if callable(getattr(Model, method))]):
                return True

        return NotImplemented

### Define the generic nodetype

In [23]:
class Node:
   
    def __init__(self, name=None):
        self.name = name or self.__class__.__name__
        
    def __repr__(self):
        return self.name
    
    def fit(self, df: pl.DataFrame, *args, **kwargs):
        raise NotImplementedError
    
    def predict(self, df: pl.DataFrame, *args, **kwargs):
        raise NotImplementedError
    
    @property
    def labels(self) -> List[Label]:
        """Returns a list of all labels that are used by the node"""
        raise NotImplementedError
    
    def train_mask_for_label(self, label, *args, **kwargs):
        raise NotImplementedError
    
    def test_mask_for_label(self, label, *args, **kwargs):
        raise NotImplementedError
    
    def get_train_df_for_label(self, df: pl.DataFrame, label, *args, **kwargs):
        return df.filter(self.train_mask_for_label(label, *args, **kwargs))
    
    def get_test_df_for_label(self, df: pl.DataFrame, label, *args, **kwargs):
        return df.filter(self.test_mask_for_label(label, *args, **kwargs))
        

### Define the simplest operation - the lift

In [24]:
class Lift(Node):

    def __init__(self,
                 child: Model,
                 atomics: Iterable[pl.DataType],
                 name: str = None,
                 ):
        name = name or f"{child.name} x {set(atomics)}"
        super().__init__(name=name)
        self.child = child
        self.models = None
        self._atomic_labels = atomics

    def train_mask_for_label(self, label, *args, **kwargs) -> pl.Expr:
        raise NotImplementedError

    def test_mask_for_label(self, label, *args, **kwargs) -> pl.Expr:
        raise NotImplementedError

    @property
    def labels(self):
        # Labels of a lift are the cartesian product of the labels
        # we are lifting to and the labels of the original model
        labels = []
        for child_label in self.child.labels:
            for atomic in self._atomic_labels:
                labels.append(
                    Label({**child_label.mapping, self.name: atomic})
                )

        return labels
        
        
        
        
                 
                 
                 

### Define ensembling

In [25]:
class Ensemble(Node):
    pass